# Controle do tamanho do discurso — regressão logística

Para cada tema, estima a probabilidade de o discurso ser sinalizado pela **R2** em cada legislatura, **descontando o efeito do tamanho do discurso** (log do número de tokens).

Modelo: `R2 ~ legislatura + log(tokens)` (regressão logística). A prevalência ajustada de cada legislatura é calculada por padronização marginal, mantendo os tamanhos reais dos discursos.

**Dados de entrada:** corpus pré-processado em `legislatura/data/artigo1_preprocessado`, gerado por `scripts/01_preprocess_corpus.py` a partir dos dados da API da Câmara dos Deputados (ver `data/README_DATA.md`). O notebook procura essa pasta no Google Drive montado.

**Execução:** Google Colab → *Ambiente de execução → Executar tudo*.

**Dicionários:** este notebook utiliza a versão completa dos dicionários. Os resultados com a versão final, adotada na dissertação, são gerados pelo notebook `23_camadas1_2_dicionario_final.ipynb`.

In [ ]:
# 1) Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Localizar a pasta pré-processada da dissertação
from pathlib import Path
RAIZ = Path('/content/drive/MyDrive')
candidatos = [RAIZ / 'legislatura' / 'data' / 'artigo1_preprocessado']
candidatos += list(RAIZ.glob('*/legislatura/data/artigo1_preprocessado'))
candidatos += list(RAIZ.glob('*/data/artigo1_preprocessado'))
PRE = next((c for c in candidatos if (c / 'corpus_artigo1_metadata.csv').exists()), None)
if PRE is None:
    raise SystemExit('Pasta artigo1_preprocessado não encontrada no Drive montado (ver Dados de entrada no topo).')
print('Pasta encontrada:', PRE)

SAIDA = RAIZ / 'MESTRADO' / 'resultados_controle_tamanho'
SAIDA.mkdir(parents=True, exist_ok=True)
PERCENTIL_R3 = 0.50   # percentil do limiar fixo da R3 (0.50 = mediana)

In [ ]:
# 3) Ler os discursos pré-processados (mesma leitura do script 14)
import pandas as pd
from collections import Counter
LEGS = [52, 53, 54, 55, 56, 57]
meta = pd.read_csv(PRE / 'corpus_artigo1_metadata.csv', usecols=['id_legislatura'])
textos, legs = [], []
for leg in LEGS:
    linhas = (PRE / 'by_legislature' / f'leg{leg}.txt').read_text(encoding='utf-8').splitlines()
    n_meta = int((meta['id_legislatura'] == leg).sum())
    if n_meta != len(linhas):
        raise ValueError(f'Leg {leg}: metadados={n_meta} texto={len(linhas)}')
    textos += linhas; legs += [leg] * len(linhas)
    print(f'Legislatura {leg}: {len(linhas):,} discursos')
df = pd.DataFrame({'_leg': legs, '_tokens': [t.split() for t in textos]})
del textos, meta
print(f'TOTAL: {len(df):,} discursos  (dissertação: 439.627)')

In [ ]:
# 4) Dicionários temáticos (idênticos aos dos scripts 08 e 14)
THEMES = {
 "Educação": {"educacao","escola","escolas","ensino","professor","professores","aluno","alunos","universidade","universidades","creche","alfabetizacao","fundeb","enem","estudante","estudantes","pesquisa","ciencia","cientifico","tecnologia"},
 "Saúde": {"saude","sus","hospital","hospitais","medico","medicos","enfermeiro","enfermeiros","enfermeiras","vacina","vacinas","pandemia","covid","doenca","doencas","tratamento","medicamento","medicamentos","pacientes","atendimento"},
 "Economia, trabalho e renda": {"economia","economico","economica","emprego","empregos","trabalho","trabalhador","trabalhadores","renda","salario","salarios","inflacao","juros","credito","industria","empresas","empresa","mercado","crescimento","desenvolvimento","investimento","investimentos","tributaria","imposto","impostos"},
 "Segurança pública e violência": {"seguranca","violencia","crime","crimes","criminalidade","policia","policial","policiais","prisao","presidio","armas","arma","trafico","homicidio","homicidios","feminicidio","militar","militares"},
 "Meio ambiente e clima": {"meio","ambiente","ambiental","amazonia","floresta","florestas","desmatamento","clima","climatica","climatico","sustentavel","sustentabilidade","biodiversidade","indigena","indigenas","garimpo","queimadas","agua","saneamento"},
 "Infraestrutura, transporte e habitação": {"infraestrutura","transporte","transportes","rodovia","rodovias","estrada","estradas","ferrovia","ferrovias","porto","portos","aeroporto","aeroportos","energia","eletrica","habitacao","moradia","saneamento","internet","telecomunicacoes","obra","obras"},
 "Proteção social e desigualdade": {"pobreza","fome","desigualdade","desigualdades","assistencia","social","bolsa","familia","beneficio","beneficios","previdencia","aposentadoria","aposentados","vulnerabilidade","moradia","direitos","mulheres","criancas","juventude","idosos","deficiencia"},
 "Federalismo e municípios": {"municipio","municipios","prefeito","prefeitos","prefeitura","prefeituras","estado","estados","federacao","federal","governador","governadores","regional","regionais","regiao","regioes","nordeste","norte","sul","sudeste","centro"},
 "Democracia e instituições": {"democracia","democratico","democratica","constituicao","constitucional","congresso","camara","senado","parlamento","governo","presidente","presidencia","republica","eleicao","eleicoes","voto","impeachment","stf","supremo","judiciario","justica"},
 "Direitos e cidadania": {"direito","direitos","cidadania","cidadao","cidadaos","igualdade","liberdade","justica","mulher","mulheres","negros","racismo","discriminacao","lgbt","indigena","indigenas","quilombolas","criancas","adolescentes"},
 "Agricultura e alimentação": {"agricultura","agricola","agropecuaria","agronegocio","produtor","produtores","rural","campo","alimentos","alimento","comida","fome","pecuaria","safra","assentamento","reforma","agraria"},
}
print(len(THEMES), 'temas')

In [ ]:
# 5) Contagem por discurso e tema
import numpy as np
rows = []
for i, toks in enumerate(df["_tokens"]):
    c = Counter(toks); n = len(toks)
    for tema, termos in THEMES.items():
        hits = [t for t in termos if c[t] > 0]
        total = sum(c[t] for t in hits)
        rows.append((i, tema, len(hits), total, 10_000 * total / n if n else 0.0))
    if i % 50_000 == 0:
        print(f"  {i:,}/{len(df):,}")
s = pd.DataFrame(rows, columns=["discurso","tema","termos_distintos","ocorrencias","densidade_10k"])
del rows
N = len(df)
print("Pronto.")

In [ ]:
# Marcar R2 por discurso e tema e montar a tabela de análise
s["R2"] = ((s["termos_distintos"] >= 2) | (s["ocorrencias"] >= 3)).astype(int)
df["n_tokens"] = df["_tokens"].map(len)
df["log_tokens"] = np.log(df["n_tokens"])
dados = s[["discurso", "tema", "R2"]].merge(df[["_leg", "log_tokens"]], left_on="discurso", right_index=True)
print("Tokens médios por discurso:")
print(df.groupby("_leg")["n_tokens"].mean().round(0))

In [ ]:
# Regressão logística por tema: R2 ~ legislatura + log(tokens)
import statsmodels.formula.api as smf
LEGS = [52, 53, 54, 55, 56, 57]
linhas = []
for tema, d in dados.groupby("tema"):
    d = d.copy()
    d["leg"] = d["_leg"].astype(str)
    m = smf.logit("R2 ~ C(leg, Treatment('52')) + log_tokens", data=d).fit(disp=0)
    or_tam = float(np.exp(m.params["log_tokens"]))
    for L in LEGS:
        cf = d.assign(leg=str(L))
        ajust = float(m.predict(cf).mean()) * 100
        bruto = float(d.loc[d["_leg"] == L, "R2"].mean()) * 100
        linhas.append({"tema": tema, "legislatura": L, "R2_bruto_%": round(bruto, 1),
                       "R2_ajustado_%": round(ajust, 1), "OR_log_tokens": round(or_tam, 2)})
    print(f"{tema}: pronto (razão de chances do log do tamanho = {or_tam:.2f})")
res = pd.DataFrame(linhas)

In [ ]:
# Resumo: variação da 52ª para a 57ª, sem e com controle do tamanho
p = res.pivot(index="tema", columns="legislatura", values=["R2_bruto_%", "R2_ajustado_%"])
resumo = pd.DataFrame({
    "bruto_52": p[("R2_bruto_%", 52)], "bruto_57": p[("R2_bruto_%", 57)],
    "ajust_52": p[("R2_ajustado_%", 52)], "ajust_57": p[("R2_ajustado_%", 57)],
})
resumo["var_bruta_pp"] = (resumo["bruto_57"] - resumo["bruto_52"]).round(1)
resumo["var_ajustada_pp"] = (resumo["ajust_57"] - resumo["ajust_52"]).round(1)
resumo = resumo.sort_values("var_bruta_pp")
resumo

In [ ]:
# Tabela completa ajustada (temas x legislaturas)
tab_ajust = res.pivot(index="tema", columns="legislatura", values="R2_ajustado_%")
tab_ajust

In [ ]:
# Salvar no Drive
res.to_csv(SAIDA / "R2_bruto_vs_ajustado_por_legislatura.csv", index=False, encoding="utf-8-sig")
resumo.to_csv(SAIDA / "resumo_variacao_52_57.csv", encoding="utf-8-sig")
tab_ajust.to_csv(SAIDA / "R2_ajustado_tamanho.csv", encoding="utf-8-sig")
print("Arquivos salvos em:", SAIDA)